<a href="https://colab.research.google.com/github/roya90/6620/blob/main/attention_from_word_counts_to_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# From Word Counts to Attention  
## Building a Tiny Language Model Step by Step

In the previous session, we learned how raw text becomes model-ready through preprocessing, tokenization, Bag-of-Words, n-grams, and TF-IDF.

Today we move from **counting words** to **learning context**.

We will build the idea step by step:

```text
text → tokens → next-token prediction → bigram model → no-attention neural model → context averaging → self-attention → multi-head attention
```

The key question for today:

```text
When understanding this token, which other tokens should matter?
```

## Learning Goals

By the end of this notebook, you should be able to explain:

1. How next-token prediction works.
2. Why a bigram model is limited.
3. Why token embeddings and position embeddings are not enough by themselves.
4. How averaging previous tokens gives a first form of context.
5. How self-attention replaces fixed averaging with learned weights.
6. What Query, Key, and Value mean.
7. Why multi-head attention is useful.
8. How attention connects to the Transformer.

## 1. Why Word Counts Are Not Enough

Classical representations such as Bag-of-Words and TF-IDF answer questions like:

```text
Which words appear?
How often do they appear?
Which words are rare but informative?
```

These are useful questions, but they are not enough for deep language understanding.

Compare:

```text
Patient denies chest pain.
Patient reports chest pain.
```

Both sentences contain many of the same words:

```text
patient
chest
pain
```

But the meanings are different.

The word `denies` changes the clinical meaning of the sentence.

## 2. Meaning Depends on Context

The same word can mean different things depending on surrounding words.

```text
The bank approved the loan.
The boat reached the river bank.
```

The token `bank` is not enough by itself.

Its meaning depends on context.

Modern language models try to learn this through a simple training task:

```text
Given previous tokens, predict the next token.
```

# Part A — Language Modeling Setup

We begin with a small dataset.

This is not a real clinical dataset. It is a teaching dataset designed so we can see every step clearly.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

In [ ]:
sentences = [
    "patient did not report chest pain.",
    "patient reports chest pain.",
    "patient denies chest pain.",
    "blood pressure is elevated.",
    "patient reports shortness of breath.",
    "patient did not report shortness of breath.",
    "kidney function is stable.",
    "patient reports joint and muscle weakness.",
    "chest x-ray was normal.",
    "patient denies dizziness or frequent coughing."
]

# Repeat the tiny corpus so the model has enough examples to learn simple patterns.
text = "\n".join(sentences * 100)

print(text[:500])
print("\nNumber of characters:", len(text))

patient did not report chest pain.
patient reports chest pain.
patient denies chest pain.
blood pressure is elevated.
patient reports shortness of breath.
patient did not report shortness of breath.
kidney function is stable.
patient reports joint and muscle weakness.
chest x-ray was normal.
patient denies dizziness or frequent coughing.
patient did not report chest pain.
patient reports chest pain.
patient denies chest pain.
blood pressure is elevated.
patient reports shortness of breath.
patie

Number of characters: 33999


## 3. Character-Level Tokenization

For simplicity, we use characters as tokens.

A real language model often uses subword tokens, but character-level tokenization is easier for learning the mechanics.

The model cannot read raw text directly.  
We first convert each character into an integer.

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

print("Vocabulary:")
print(chars)
print("\nVocabulary size:", vocab_size)

Vocabulary:
['\n', ' ', '-', '.', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']

Vocabulary size: 30


In [ ]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join([itos[i] for i in ids])

sample = "patient reports chest pain."
encoded = encode(sample)

print("Original text:")
print(sample)

print("\nEncoded:")
print(encoded)

print("\nDecoded:")
print(decode(encoded))

Original text:
patient reports chest pain.

Encoded:
[19, 4, 23, 12, 8, 17, 23, 1, 21, 8, 19, 18, 21, 23, 22, 1, 6, 11, 8, 22, 23, 1, 19, 4, 12, 17, 3]

Decoded:
patient reports chest pain.


## 4. Token IDs Are Just Labels

A token ID is not meaning.

For example, the model may store:

```text
p → 14
a → 3
t → 18
```

But the number `18` does not mean that `t` is larger, better, or more important than `a`.

The integer is just an identifier.

The model must learn useful representations from these IDs.

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)

print(data[:80])
print("\nDecoded:")
print(decode(data[:80].tolist()))
print("\nData shape:", data.shape)

tensor([19,  4, 23, 12,  8, 17, 23,  1,  7, 12,  7,  1, 17, 18, 23,  1, 21,  8,
        19, 18, 21, 23,  1,  6, 11,  8, 22, 23,  1, 19,  4, 12, 17,  3,  0, 19,
         4, 23, 12,  8, 17, 23,  1, 21,  8, 19, 18, 21, 23, 22,  1,  6, 11,  8,
        22, 23,  1, 19,  4, 12, 17,  3,  0, 19,  4, 23, 12,  8, 17, 23,  1,  7,
         8, 17, 12,  8, 22,  1,  6, 11])

Decoded:
patient did not report chest pain.
patient reports chest pain.
patient denies ch

Data shape: torch.Size([33999])


## 5. Train and Validation Split

We split the sequence into training and validation parts.

The model learns from the training data.  
The validation data gives us a rough sense of whether the model is improving beyond memorizing one batch.

In [ ]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print("Train length:", len(train_data))
print("Validation length:", len(val_data))

Train length: 30599
Validation length: 3400


## 6. Input and Target

Language modeling uses a shifted target.

If the input is:

```text
patient
```

the target is:

```text
atient...
```

At every position, the model predicts the next token.

In [ ]:
block_size = 16

x = train_data[:block_size]
y = train_data[1:block_size + 1]

print("Input IDs:")
print(x)

print("\nTarget IDs:")
print(y)

print("\nInput decoded:")
print(decode(x.tolist()))

print("\nTarget decoded:")
print(decode(y.tolist()))

Input IDs:
tensor([19,  4, 23, 12,  8, 17, 23,  1,  7, 12,  7,  1, 17, 18, 23,  1])

Target IDs:
tensor([ 4, 23, 12,  8, 17, 23,  1,  7, 12,  7,  1, 17, 18, 23,  1, 21])

Input decoded:
patient did not 

Target decoded:
atient did not r


## 7. One Block Contains Many Training Examples

Inside one block, every position creates a prediction task.

For example:

```text
p → a
pa → t
pat → i
pati → e
```

The model learns from each position.

In [ ]:
for t in range(8):
    context = x[:t+1]
    target = y[t]
    print(f"context: {decode(context.tolist())!r:12s} → target: {decode([target.item()])!r}")

context: 'p'          → target: 'a'
context: 'pa'         → target: 't'
context: 'pat'        → target: 'i'
context: 'pati'       → target: 'e'
context: 'patie'      → target: 'n'
context: 'patien'     → target: 't'
context: 'patient'    → target: ' '
context: 'patient '   → target: 'd'


## 8. Batches

A batch gives the model multiple examples at once.

Shape:

```text
batch_size × block_size
```

Each row is one sequence chunk.

In [ ]:
batch_size = 4


def get_batch(split):
    source = train_data if split == "train" else val_data
    ix = torch.randint(len(source) - block_size, (batch_size,))
    x = torch.stack([source[i:i+block_size] for i in ix])
    y = torch.stack([source[i+1:i+block_size+1] for i in ix])
    return x, y


xb, yb = get_batch("train")

print("Input batch shape:", xb.shape)
print("Target batch shape:", yb.shape)

Input batch shape: torch.Size([4, 16])
Target batch shape: torch.Size([4, 16])


In [ ]:
for b in range(3):
    print("input: ", decode(xb[b].tolist()))
    print("target:", decode(yb[b].tolist()))
    print()

input:  normal.
patient 
target: ormal.
patient d

input:  stable.
patient 
target: table.
patient r

input:  hing.
patient di
target: ing.
patient did



# Part B — First Model: Bigram Language Model

Now we build the simplest possible language model.

A **bigram model** predicts the next token using only the current token.

```text
current token → next token
```

Example:

```text
p → a
a → t
t → i
```

This model has no attention and no real context.

## 9. Logits

The model outputs **logits**.

Logits are raw scores before softmax.

For one token position, the model gives a score for every possible next token.

Example:

```text
current token: p

score for a: high
score for z: low
score for space: medium
```

In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        # Each token directly maps to scores for the next token.
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx shape: B × T
        # logits shape: B × T × vocab_size
        logits = self.token_embedding_table(idx)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            loss = F.cross_entropy(
                logits.reshape(B * T, C),
                targets.reshape(B * T)
            )

        return logits, loss

In [ ]:
bigram_model = BigramLanguageModel(vocab_size)

logits, loss = bigram_model(xb, yb)

print("Logits shape:", logits.shape)
print("Initial loss:", loss.item())

Logits shape: torch.Size([4, 16, 30])
Initial loss: 3.7114274501800537


## 10. Generation

Generation means:

```text
predict next token
append it
predict next token again
repeat
```

The model produces text one token at a time.

In [ ]:
@torch.no_grad()
def generate(model, idx, max_new_tokens):
    model.eval()

    for _ in range(max_new_tokens):
        # Crop to the most recent block_size tokens.
        # This is necessary for models that use position embeddings.
        idx_cond = idx[:, -block_size:]

        logits, loss = model(idx_cond)

        # Focus only on the last time step.
        logits = logits[:, -1, :]

        # Convert logits to probabilities.
        probs = F.softmax(logits, dim=-1)

        # Sample one token.
        next_id = torch.multinomial(probs, num_samples=1)

        # Append the sampled token.
        idx = torch.cat((idx, next_id), dim=1)

    model.train()
    return idx

In [ ]:
start = torch.zeros((1, 1), dtype=torch.long)

generated = generate(bigram_model, start, max_new_tokens=200)

print(decode(generated[0].tolist()))


qzikxag
kvbr-tfluyk-mvbdxqzwgkclkpqco..y
gefmzic
kej.wcdfzvhappfao
mphxidb xnnsvbrjmluuyrxfi..a
yebbpu-mpxqfggypf-smkagkebbrs
hlus
hulie mnwkrltdfwvbsgke.hiuy pinnvt jbiualtr
zkpgafagnnltxzius..csmpus


Before training, the output looks random.

That is expected.

The model has not learned token patterns yet.

## 11. Training Helper

We will reuse the same training function for multiple models.

The training loop follows this structure:

```text
sample a batch
run the model
calculate loss
zero gradients
backpropagate
update parameters
repeat
```

In [ ]:
@torch.no_grad()
def estimate_loss(model, eval_iters=5):
    model.eval()
    out = {}

    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)

        for k in range(eval_iters):
            xb, yb = get_batch(split)
            logits, loss = model(xb, yb)
            losses[k] = loss.item()

        out[split] = losses.mean().item()

    model.train()
    return out


def train_model(model, max_iters=120, learning_rate=1e-2, eval_interval=40):
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

    for step in range(max_iters + 1):
        if step % eval_interval == 0:
            losses = estimate_loss(model)
            print(
                f"step {step:4d} | "
                f"train loss {losses['train']:.4f} | "
                f"val loss {losses['val']:.4f}"
            )

        xb, yb = get_batch("train")
        logits, loss = model(xb, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return model

In [ ]:
bigram_model = BigramLanguageModel(vocab_size)
bigram_model = train_model(
    bigram_model,
    max_iters=120,
    learning_rate=1e-2,
    eval_interval=40
)

step    0 | train loss 3.7148 | val loss 3.7179
step   40 | train loss 3.2989 | val loss 3.3626
step   80 | train loss 2.7994 | val loss 2.8574
step  120 | train loss 2.6811 | val loss 2.5861


In [ ]:
generated = generate(bigram_model, start, max_new_tokens=300)
print(decode(generated[0].tolist()))


xg.iddb.-jpjwwtth.rnvwfqinwejncxpiscvrloyjpc..qgakmgbkr
tst
pcxo-pf-
p cj
cvyhbiogt qvdebgv h ju
pi-jnwwpwe-j
pisui-jndiccqi-we.z
.fre-delo-nchizt
kon i.izfqgporakxycxz
t
plounfzxdelmm.xedescqghomylxycddovwh.-lscbloovbk-jwoycbwjpi.zzt bdnt pz
pqgbwocj
pvcjnmlmigi-qakgtgcjwgainfqd rmwfblnt gdnwtshapx


## 12. Bigram Model Limitation

The bigram model only sees the current token.

It does not see:

```text
previous tokens
full phrases
sentence structure
negation
long-distance context
```

Example:

```text
patient did not report chest pain
```

The meaning of `pain` depends on earlier tokens like:

```text
not
report
chest
```

The bigram model cannot use them.

# Part C — A Neural Model Without Attention

Before adding attention, we build a slightly better neural language model.

This model has:

```text
token embeddings
position embeddings
prediction head
```

But it still has no attention.

This is important: the model knows **what** token it sees and **where** it is, but tokens still do not communicate with each other.

In [ ]:
n_embd = 32


class NoAttentionLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embd, block_size):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)  # B × T × n_embd
        pos_emb = self.position_embedding_table(torch.arange(T))  # T × n_embd

        x = tok_emb + pos_emb  # B × T × n_embd

        logits = self.lm_head(x)  # B × T × vocab_size

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            loss = F.cross_entropy(
                logits.reshape(B * T, C),
                targets.reshape(B * T)
            )

        return logits, loss

In [ ]:
no_attention_model = NoAttentionLanguageModel(vocab_size, n_embd, block_size)

logits, loss = no_attention_model(xb, yb)

print("Logits shape:", logits.shape)
print("Initial loss:", loss.item())

Logits shape: torch.Size([4, 16, 30])
Initial loss: 3.759803295135498


In [ ]:
no_attention_model = train_model(
    no_attention_model,
    max_iters=120,
    learning_rate=1e-2,
    eval_interval=40
)

step    0 | train loss 3.7894 | val loss 3.7397
step   40 | train loss 2.0000 | val loss 2.1370
step   80 | train loss 1.8017 | val loss 1.7608
step  120 | train loss 1.8433 | val loss 1.6402


In [ ]:
generated = generate(no_attention_model, start, max_new_tokens=300)
print(decode(generated[0].tolist()))


bleyhe r of poley nches bltodeytidaidnthepabrtish.
kieykiny wrtshoesd joreaidnean.
patsthesssstst futnesth.
brtn.
pabrenn.
pofutincblel.
bl.
cofust d d cinkbl jort .
brentinctinent d jorevkn.
brt g ne jt dn.
kijkneymx- bst zot joressd ch.
knt x-rmatintin.
brent bry wntiothokst bledncy jorthel.
portn


## 13. What Is Still Missing?

This model has:

```text
token identity
token position
output prediction layer
```

But it still does not have:

```text
communication between tokens
```

At each position, the model knows:

```text
what token am I?
where am I?
```

But it does not know:

```text
which previous tokens matter to me?
```

That is the missing idea.

# Part D — First Step Toward Attention: Averaging the Past

We want each token representation to include information from earlier tokens.

Instead of:

```text
token alone
```

we want:

```text
token + previous context
```

The simplest idea is:

```text
average all previous token representations
```

This is not attention yet.

But it is the first step toward attention.

In [ ]:
torch.manual_seed(42)

B_demo, T_demo, C_demo = 4, 8, 2
x_demo = torch.randn(B_demo, T_demo, C_demo)

print("x_demo shape:", x_demo.shape)
print("\nFirst sequence:")
print(x_demo[0])

x_demo shape: torch.Size([4, 8, 2])

First sequence:
tensor([[ 1.9269,  1.4873],
        [ 0.9007, -2.1055],
        [ 0.6784, -1.2345],
        [-0.0431, -1.6047],
        [-0.7521,  1.6487],
        [-0.3925, -1.4036],
        [-0.7279, -0.5594],
        [-0.7688,  0.7624]])


## 14. Average Previous Tokens with Loops

For each position, we average all previous token vectors including the current token.

At position 0:

```text
use token 0
```

At position 1:

```text
average token 0 and token 1
```

At position 2:

```text
average token 0, token 1, and token 2
```

In [ ]:
xbow = torch.zeros((B_demo, T_demo, C_demo))

for b in range(B_demo):
    for t in range(T_demo):
        xprev = x_demo[b, :t+1]
        xbow[b, t] = torch.mean(xprev, dim=0)

print("Original first sequence:")
print(x_demo[0])

print("\nAveraged-with-history first sequence:")
print(xbow[0])

Original first sequence:
tensor([[ 1.9269,  1.4873],
        [ 0.9007, -2.1055],
        [ 0.6784, -1.2345],
        [-0.0431, -1.6047],
        [-0.7521,  1.6487],
        [-0.3925, -1.4036],
        [-0.7279, -0.5594],
        [-0.7688,  0.7624]])

Averaged-with-history first sequence:
tensor([[ 1.9269,  1.4873],
        [ 1.4138, -0.3091],
        [ 1.1687, -0.6176],
        [ 0.8657, -0.8644],
        [ 0.5422, -0.3617],
        [ 0.3864, -0.5354],
        [ 0.2272, -0.5388],
        [ 0.1027, -0.3762]])


This gives each position a summary of the past.

But the loop is slow.

We can do the same operation with matrix multiplication.

## 15. Matrix Trick for Averaging

We create a lower triangular matrix.

This matrix controls which tokens are allowed to communicate.

In [ ]:
wei = torch.tril(torch.ones(T_demo, T_demo))
wei = wei / wei.sum(dim=1, keepdim=True)

print(wei)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])


In [ ]:
xbow2 = wei @ x_demo

print(torch.allclose(xbow, xbow2))

True


The lower triangular structure means:

```text
token 0 can see token 0
token 1 can see token 0 and token 1
token 2 can see token 0, token 1, and token 2
```

Tokens cannot see the future.

This is called **causal masking**.

## 16. Softmax Version

Now we rewrite the same averaging operation using masking and softmax.

This gets us closer to attention.

In [ ]:
tril = torch.tril(torch.ones(T_demo, T_demo))

wei = torch.zeros((T_demo, T_demo))
wei = wei.masked_fill(tril == 0, float("-inf"))
wei = F.softmax(wei, dim=-1)

print(wei)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])


In [ ]:
xbow3 = wei @ x_demo

print(torch.allclose(xbow, xbow3))

True


## 17. Why Averaging Is Not Enough

Averaging treats all previous tokens equally.

But language does not work that way.

In:

```text
patient did not report chest pain
```

when understanding `pain`, some previous tokens are more important than others:

```text
not
report
chest
```

So we need learned weights, not fixed averaging.

That is attention.

# Part E — Self-Attention

Attention replaces fixed averaging with learned importance.

Instead of:

```text
all previous tokens contribute evenly
```

we want:

```text
the model learns which previous tokens matter more
```

Attention lets each token decide where to look.

## 18. Query, Key, and Value

Each token creates three vectors:

```text
Query: what am I looking for?
Key: what information do I contain?
Value: what information do I pass forward?
```

Analogy:

```text
Query = question
Key = label on a folder
Value = contents inside the folder
```

The query searches through keys.

The matching values are combined.

In [ ]:
torch.manual_seed(42)

B_demo, T_demo, C_demo = 4, 8, 32
x_demo = torch.randn(B_demo, T_demo, C_demo)

head_size = 16

key = nn.Linear(C_demo, head_size, bias=False)
query = nn.Linear(C_demo, head_size, bias=False)
value = nn.Linear(C_demo, head_size, bias=False)

k = key(x_demo)
q = query(x_demo)
v = value(x_demo)

print("q shape:", q.shape)
print("k shape:", k.shape)
print("v shape:", v.shape)

q shape: torch.Size([4, 8, 16])
k shape: torch.Size([4, 8, 16])
v shape: torch.Size([4, 8, 16])


## 19. Compute Attention Scores

We compare queries with keys.

The result has shape:

```text
B × T × T
```

For every sequence in the batch, each token gets a relevance score with every other token.

In [ ]:
wei = q @ k.transpose(-2, -1)

print("Attention score shape:", wei.shape)
print("\nScores for the first sequence:")
print(wei[0])

Attention score shape: torch.Size([4, 8, 8])

Scores for the first sequence:
tensor([[-0.3332, -1.1723, -1.0216, -0.0545, -1.0950,  0.2735,  0.1340, -0.8490],
        [-0.6597,  0.7869, -1.2725,  1.6851,  0.1159,  0.5450,  0.2356, -0.1962],
        [ 0.3630, -1.5219,  0.7821, -1.7215, -0.3494,  0.2884, -0.1021, -1.4271],
        [-0.1001,  0.8649, -0.0335,  1.0221, -0.1350, -0.3078,  0.1440, -0.3019],
        [ 0.0136, -1.6202, -1.9888, -0.3327, -1.2507, -0.8928, -2.2674,  3.0561],
        [-0.5833,  1.2025, -0.3281,  0.9147,  0.9809, -0.4859,  1.7589,  0.1650],
        [ 1.1351, -1.9940,  1.5545, -1.8037, -0.5062, -2.6109, -1.0739,  1.6430],
        [-1.2784, -0.4554, -1.4118,  0.6392, -0.5780,  1.9291,  1.6689,  0.1103]],
       grad_fn=<SelectBackward0>)


A high score means:

```text
this token is relevant
```

A low score means:

```text
this token is less relevant
```

The model learns these patterns during training.

## 20. Scale the Scores

Attention uses scaled dot products.

```text
QKᵀ / √d_k
```

Scaling keeps the scores from becoming too large before softmax.

In [ ]:
wei = wei * head_size**-0.5

print(wei[0])

tensor([[-0.0833, -0.2931, -0.2554, -0.0136, -0.2738,  0.0684,  0.0335, -0.2122],
        [-0.1649,  0.1967, -0.3181,  0.4213,  0.0290,  0.1363,  0.0589, -0.0490],
        [ 0.0907, -0.3805,  0.1955, -0.4304, -0.0874,  0.0721, -0.0255, -0.3568],
        [-0.0250,  0.2162, -0.0084,  0.2555, -0.0338, -0.0769,  0.0360, -0.0755],
        [ 0.0034, -0.4050, -0.4972, -0.0832, -0.3127, -0.2232, -0.5669,  0.7640],
        [-0.1458,  0.3006, -0.0820,  0.2287,  0.2452, -0.1215,  0.4397,  0.0412],
        [ 0.2838, -0.4985,  0.3886, -0.4509, -0.1266, -0.6527, -0.2685,  0.4107],
        [-0.3196, -0.1138, -0.3529,  0.1598, -0.1445,  0.4823,  0.4172,  0.0276]],
       grad_fn=<SelectBackward0>)


## 21. Apply Causal Mask

For GPT-style generation, tokens cannot look into the future.

We use the same lower triangular mask.

In [ ]:
tril = torch.tril(torch.ones(T_demo, T_demo))

wei = wei.masked_fill(tril == 0, float("-inf"))

print(wei[0])

tensor([[-0.0833,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.1649,  0.1967,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 0.0907, -0.3805,  0.1955,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.0250,  0.2162, -0.0084,  0.2555,    -inf,    -inf,    -inf,    -inf],
        [ 0.0034, -0.4050, -0.4972, -0.0832, -0.3127,    -inf,    -inf,    -inf],
        [-0.1458,  0.3006, -0.0820,  0.2287,  0.2452, -0.1215,    -inf,    -inf],
        [ 0.2838, -0.4985,  0.3886, -0.4509, -0.1266, -0.6527, -0.2685,    -inf],
        [-0.3196, -0.1138, -0.3529,  0.1598, -0.1445,  0.4823,  0.4172,  0.0276]],
       grad_fn=<SelectBackward0>)


## 22. Convert Scores to Weights

Softmax converts scores into attention weights.

Each row sums to 1.

Each row answers:

```text
For this token, how much attention should go to each previous token?
```

In [ ]:
wei = F.softmax(wei, dim=-1)

print(wei[0])
print("\nRow sums:")
print(wei[0].sum(dim=-1))

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4106, 0.5894, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3657, 0.2283, 0.4061, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2168, 0.2759, 0.2204, 0.2870, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2553, 0.1697, 0.1548, 0.2341, 0.1861, 0.0000, 0.0000, 0.0000],
        [0.1318, 0.2060, 0.1405, 0.1917, 0.1949, 0.1351, 0.0000, 0.0000],
        [0.2137, 0.0978, 0.2374, 0.1025, 0.1418, 0.0838, 0.1230, 0.0000],
        [0.0852, 0.1047, 0.0824, 0.1376, 0.1015, 0.1900, 0.1780, 0.1206]],
       grad_fn=<SelectBackward0>)

Row sums:
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
       grad_fn=<SumBackward1>)


## 23. Weighted Sum of Values

Now we use the attention weights to combine the value vectors.

In [ ]:
out = wei @ v

print("Output shape:", out.shape)

Output shape: torch.Size([4, 8, 16])


This is self-attention.

Each token receives a new representation that mixes information from relevant previous tokens.

Before attention:

```text
token representation = token alone
```

After attention:

```text
token representation = token + learned context
```

## 24. The Attention Formula

Now the formula has meaning:

```text
Attention(Q, K, V) = softmax(QKᵀ / √d_k) V
```

Step by step:

```text
1. Create Query, Key, and Value
2. Compare Query with Key
3. Scale the scores
4. Mask future tokens
5. Softmax into weights
6. Use weights to combine Values
```

# Part F — Package One Attention Head

Now we turn the attention logic into a reusable class.

One attention head is one learned way of looking at context.

In [ ]:
class Head(nn.Module):
    def __init__(self, n_embd, head_size, block_size):
        super().__init__()

        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer(
            "tril",
            torch.tril(torch.ones(block_size, block_size))
        )

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2, -1)
        wei = wei * k.shape[-1]**-0.5

        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)

        v = self.value(x)
        out = wei @ v

        return out

In [ ]:
head = Head(n_embd=32, head_size=16, block_size=block_size)

x_test = torch.randn(4, block_size, 32)
out = head(x_test)

print(out.shape)

torch.Size([4, 16, 16])


# Part G — Add Attention to the Language Model

Now we add one self-attention head to the language model.

The model flow becomes:

```text
token IDs
→ token embeddings
→ position embeddings
→ self-attention
→ output logits
→ next-token prediction
```

Now tokens can communicate.

In [ ]:
class AttentionLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embd, block_size):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        self.sa_head = Head(n_embd, n_embd, block_size)

        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T))

        x = tok_emb + pos_emb
        x = self.sa_head(x)

        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            loss = F.cross_entropy(
                logits.reshape(B * T, C),
                targets.reshape(B * T)
            )

        return logits, loss

In [ ]:
attention_model = AttentionLanguageModel(vocab_size, n_embd, block_size)

logits, loss = attention_model(xb, yb)

print("Logits shape:", logits.shape)
print("Initial loss:", loss.item())

Logits shape: torch.Size([4, 16, 30])
Initial loss: 3.4032411575317383


In [ ]:
attention_model = train_model(
    attention_model,
    max_iters=120,
    learning_rate=1e-2,
    eval_interval=40
)

step    0 | train loss 3.4368 | val loss 3.4238
step   40 | train loss 2.3075 | val loss 2.0114
step   80 | train loss 1.9283 | val loss 1.6604
step  120 | train loss 1.6136 | val loss 1.8651


In [ ]:
generated = generate(attention_model, start, max_new_tokens=300)
print(decode(generated[0].tolist()))


t cotesst res orenin rtint muentiatirf s chortiinorst rtughorshelevat rirt briort preld in.
prieoren.
paiintiint orent f iedefuresut de wevat d ingheshentienevat dnor-r fucht did.
pat nmatid risheqchn.
bredin.
porepats lewieninchen.
katinent oresent dnieltinororesshieysssched surt pot deyughofughie 


## 25. What Changed?

The bigram model did this:

```text
current token → next token
```

The no-attention model did this:

```text
token identity + token position → next token
```

The attention model does this:

```text
token identity + token position + relevant previous tokens → next token
```

The key upgrade is communication.

Attention lets tokens exchange information.

# Part H — Multi-Head Attention

One attention head gives the model one way to look at context.

But language has many types of relationships:

```text
negation
nearby words
phrase structure
subject-verb relationships
long-distance dependencies
domain-specific terms
```

Multi-head attention lets the model learn several attention patterns in parallel.

## 26. Multi-Head Attention Intuition

Analogy:

```text
Several students read the same sentence.

One highlights negation.
One highlights clinical terms.
One highlights grammar.
One highlights long-distance dependencies.
```

Then the model combines their views.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, num_heads, head_size, block_size):
        super().__init__()

        self.heads = nn.ModuleList([
            Head(n_embd, head_size, block_size)
            for _ in range(num_heads)
        ])

        self.proj = nn.Linear(num_heads * head_size, n_embd)

    def forward(self, x):
        out = torch.cat([head(x) for head in self.heads], dim=-1)
        out = self.proj(out)
        return out

In [ ]:
num_heads = 4
head_size = n_embd // num_heads

multi_head = MultiHeadAttention(n_embd, num_heads, head_size, block_size)

x_test = torch.randn(4, block_size, n_embd)
out = multi_head(x_test)

print(out.shape)

torch.Size([4, 16, 32])


# Part I — Multi-Head Attention Language Model

Now we replace the single attention head with multi-head attention.

In [ ]:
class MultiHeadLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embd, block_size, num_heads):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        head_size = n_embd // num_heads
        self.sa = MultiHeadAttention(
            n_embd=n_embd,
            num_heads=num_heads,
            head_size=head_size,
            block_size=block_size
        )

        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T))

        x = tok_emb + pos_emb
        x = self.sa(x)

        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            loss = F.cross_entropy(
                logits.reshape(B * T, C),
                targets.reshape(B * T)
            )

        return logits, loss

In [ ]:
multi_head_model = MultiHeadLanguageModel(
    vocab_size=vocab_size,
    n_embd=n_embd,
    block_size=block_size,
    num_heads=4
)

logits, loss = multi_head_model(xb, yb)

print("Logits shape:", logits.shape)
print("Initial loss:", loss.item())

Logits shape: torch.Size([4, 16, 30])
Initial loss: 3.4195761680603027


In [ ]:
multi_head_model = train_model(
    multi_head_model,
    max_iters=120,
    learning_rate=1e-2,
    eval_interval=40
)

step    0 | train loss 3.4070 | val loss 3.3815
step   40 | train loss 2.1386 | val loss 2.0291
step   80 | train loss 1.7934 | val loss 1.7837
step  120 | train loss 1.7775 | val loss 1.5413


In [ ]:
generated = generate(multi_head_model, start, max_new_tokens=300)
print(decode(generated[0].tolist()))


dhiden.
c ch.
patienieencgregh.
patstien.
patieney re ienesquenoenesnor.
patntft atizes chequguodtn.
p
resures pathoff recghodienor.
pat dtieneft rof dge.
p
roeat.
pat.
presats reseprthenoreatequgorrte prelreres res ctesusaleves weneloeporgeaprtesres katienehevre a.
path.
patien.
relenevatinesledes 


# Part J — Connecting Attention to Transformers

A Transformer language model usually contains:

```text
token embeddings
position embeddings
self-attention
feed-forward layers
residual connections
layer normalization
output prediction head
```

Today we focused on the central idea:

```text
self-attention
```

The full Transformer stacks attention blocks many times.

## 27. Simplified Transformer Flow

```text
tokens
→ token embeddings
→ positional embeddings
→ self-attention
→ feed-forward network
→ output logits
→ next-token prediction
```

Attention is the part that lets tokens communicate.

The feed-forward layers further process each token representation.

Stacking many blocks makes the model more powerful.

## 28. Final Summary

Today’s path:

```text
1. Text becomes tokens
2. Tokens become IDs
3. IDs become embeddings
4. A bigram model predicts the next token from the current token
5. A no-attention neural model adds token and position embeddings
6. The model still lacks token communication
7. Averaging gives a first form of context
8. Self-attention learns which previous tokens matter
9. Multi-head attention learns multiple relationship patterns
10. Transformers stack attention blocks
```

The core idea:

```text
Attention lets each token learn what to focus on.
```

## 29. Reflection Questions

1. Why is a bigram model limited?
2. Why are token embeddings and position embeddings not enough by themselves?
3. What does causal masking prevent?
4. What is the difference between averaging the past and attention?
5. What are Query, Key, and Value?
6. Why might multiple attention heads be better than one?
7. How does attention create context-aware representations?